# FlashAttention in depth: the CPU companion

**Tier:** T0. numpy only, no GPU, runs in a minute on a laptop or Colab CPU.

Companion to [`flash-attention-deep-dive.md`](flash-attention-deep-dive.md). The derived numbers of that page are
computed by [`fa_calculators.py`](fa_calculators.py) (pinned by `test_fa_calculators.py`); this notebook imports it,
reruns every algorithm against a naive reference, and prints the page's numbers (section 8 collects the ones the
worked examples do not print on the way).

**Exercises.** Six cells marked **Exercise** ask you to implement a key piece or predict a number before the
notebook shows it. Replace the `None`s. Until you do, the check under each one prints `[ ]` and the notebook still
runs to the end; once you fill it in, the check asserts, and a wrong answer stops with the expected value.
Reference implementations live in `fa_calculators.py` and in the worked cells that follow each exercise.

**The one-minute version.** After running this notebook you should be able to show, with numbers, that:

1. naive attention is memory-bound at any length because its intensity tends to `d/b`;
2. the online-softmax state `(m, l, o)` merges exactly in any order, which is what tiling, split-KV decode and cascade attention all rely on;
3. an FA2-order tiled forward pass with causal block skipping matches the naive result and visits about half the tiles;
4. the backward pass needs only `O` and the log-sum-exp, not `P`;
5. split-KV decode with GQA packing is exact and reads K/V once per KV head, and how many splits a library picks;
6. FP8 errors come from mantissa rounding, outlier channels, outlier tokens and small probabilities, and each has a different fix (a rotation helps e4m3 only when the outlier channels of Q and K line up);
7. paged K/V, bottom-right causal masks and cascade attention are the same algebra with different bookkeeping.

In [ ]:
import math
import numpy as np
import fa_calculators as fc

np.set_printoptions(precision=6, suppress=True)


def check(name, ok):
    assert ok, name
    print(f"PASS  {name}")


def grade(name, got, want, rel=1e-3):
    """Exercise check: [ ] while the answer is None, then an assert with the expected value."""
    if got is None:
        print(f"[ ]   {name}: not attempted yet")
        return
    ok = bool(np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=rel, atol=1e-12))
    assert ok, f"{name}: you have {got}, expected {want}"
    print(f"PASS  {name}")

## 1. Naive vs tiled attention on the roofline (deep dive, sections 1 and 3.4)

One head, `d = 128`, bf16. Naive traffic is `4N²b + 4Ndb`; the FA2 schedule without any L2 reuse re-reads K and V once
per Q block; the compulsory traffic reads Q, K, V once and writes O once.

In [ ]:
d, b = 128, 2
print(f"{'N':>7} {'naive MB':>9} {'I':>6} {'H100 mem us':>12} {'H100 comp us':>13} {'L4 mem us':>10} "
      f"{'L4 comp us':>11} {'FA2 no-L2 MB':>13} {'compulsory MB':>14}")
for N in (1024, 4096, 32768):
    t = fc.naive_traffic(N, d, b)
    h, l4 = fc.roofline(t["flops"], t["bytes"], "H100"), fc.roofline(t["flops"], t["bytes"], "L4")
    f2 = fc.flash_traffic(N, d, 128, 64, b)
    print(f"{N:>7} {t['bytes']/1e6:>9.1f} {t['intensity']:>6.1f} {h['t_memory_s']*1e6:>12.1f} "
          f"{h['t_compute_s']*1e6:>13.1f} {l4['t_memory_s']*1e6:>10.1f} {l4['t_compute_s']*1e6:>11.1f} "
          f"{f2['bytes']/1e6:>13.1f} {fc.compulsory_bytes(N, d, b)/1e6:>14.2f}")

for dev in ("T4", "L4", "A100", "H100", "B200"):
    print(f"{dev:>5}: ridge {fc.DEVICES[dev].ridge:6.1f} FLOP/B")

check("naive intensity tends to d/b = 64", abs(fc.naive_traffic(1 << 20, d, b)["intensity"] - d / b) < 0.1)
check("FA2 tile intensity is ~B_r FLOP/B in bf16 when K/V come from DRAM",
      abs(fc.flash_traffic(1 << 18, d, 128, 64, b)["intensity"] - 128) < 1.5)

### Exercise 1: predict the naive traffic

Before running anything: one head, `N = 8,192`, `d = 64`, bf16, the three-kernel schedule of deep dive section 1.2.
How many bytes cross HBM, and what is the arithmetic intensity? Is it memory-bound on an H100 (ridge 295)?
Work it from `4N²b + 4Ndb` and `4N²d` FLOPs.

In [ ]:
naive_bytes = None        # e.g. 5.4e8
naive_intensity = None    # FLOP per byte

t = fc.naive_traffic(8192, 64)
grade("naive bytes, N = 8,192, d = 64", naive_bytes, t["bytes"], rel=0.01)
grade("naive intensity, N = 8,192, d = 64", naive_intensity, t["intensity"], rel=0.01)

## 2. The softmax state is a monoid (deep dive, section 2)

First the hand-worked example of section 2.6: one query row, `τ = 0.5`, raw scores `[4, 8]` then `[6, 12]`,
values `v1..v4`. Then the same result as a split-KV merge in LSE form, and with an FA4-style lagging max.

In [ ]:
S_blocks = [[4.0, 8.0], [6.0, 12.0]]
V_blocks = [[[1, 0], [0, 1]], [[1, 1], [2, -1]]]
tau = 0.5

ref_o, ref_lse = fc.reference_attention_row(np.array([4, 8, 6, 12.0]),
                                            np.array([[1, 0], [0, 1], [1, 1], [2, -1.0]]), tau)
print("reference  O =", ref_o, "  LSE =", round(ref_lse, 6))
for use_exp2 in (False, True):
    steps, o, lse = fc.online_trace(S_blocks, V_blocks, use_exp2=use_exp2, scale=tau)
    print("\nbase 2 (one FFMA + one EX2 per score)" if use_exp2 else "\nnatural exp")
    for i, s in enumerate(steps, 1):
        print(f"  block {i}: m={s['m']:.0f}  alpha={s['alpha']:.6f}  p={s['p']}  l={s['l']:.6f}  o={s['o_unnorm']}")
    print("  O =", o, "  LSE =", round(lse, 6))
    check("online softmax == reference" + (" (exp2)" if use_exp2 else ""),
          np.allclose(o, ref_o, atol=1e-12) and abs(lse - ref_lse) < 1e-12)

st1 = fc.block_state(tau * np.array(S_blocks[0]), np.array(V_blocks[0], float))
st2 = fc.block_state(tau * np.array(S_blocks[1]), np.array(V_blocks[1], float))
(o1, l1), (o2, l2) = st1.finalize(), st2.finalize()
o, lse = fc.merge_lse(o1, l1, o2, l2)
print(f"\nsplit 1: O={o1} LSE={l1:.6f}   split 2: O={o2} LSE={l2:.6f}")
print(f"merged : O={o} LSE={lse:.6f}   weights {math.exp(l1 - lse):.6f}, {math.exp(l2 - lse):.6f}")
check("LSE-form merge == reference", np.allclose(o, ref_o, atol=1e-12) and abs(lse - ref_lse) < 1e-12)

c = tau * fc.LOG2E
growth = (12 - 8) * c                      # how far the max moved, in log2 units
m_ref = 8.0 if growth < 3.0 else 12.0      # threshold 3 for illustration; FA4 uses 8
p1 = 2 ** (np.array(S_blocks[0]) * c - m_ref * c)
p2 = 2 ** (np.array(S_blocks[1]) * c - m_ref * c)
o_lag = p1 @ np.array(V_blocks[0], float) + p2 @ np.array(V_blocks[1], float)
l_lag = p1.sum() + p2.sum()
print(f"\nmax grew by {growth:.3f} log2 units -> keep m = {m_ref:.0f}; p2 = {p2}; l = {l_lag:.6f}; O = {o_lag / l_lag}")
check("a lagging reference max gives the same O", np.allclose(o_lag / l_lag, ref_o, atol=1e-12))

### Exercise 2: write the merge

Implement `my_merge(a, b)` for two `fc.State(m, l, o)` partial results built from disjoint sets of keys (deep dive
section 2.3). The scores are already scaled, so the weights are `exp(m_a − m)` and `exp(m_b − m)`. Guard the empty
state `(−∞, 0, 0)`: merging two of them must give `l = 0` and no NaN, because `(−∞) − (−∞)` is NaN.

In [ ]:
def my_merge(a, b):
    """Return fc.State(m, l, o) for the union of the keys behind a and b."""
    # YOUR CODE: shared reference max, two weights (0 for an empty side), then add
    return None


probe = my_merge(fc.empty_state(2), fc.empty_state(2))
if probe is None:
    print("[ ]   my_merge: not attempted yet")
else:
    rng = np.random.default_rng(11)
    sc, vv = rng.normal(size=10) * 3, rng.normal(size=(10, 4))
    parts = [fc.block_state(sc[i:i + 3], vv[i:i + 3]) for i in (0, 3, 6)] + [fc.block_state(sc[9:], vv[9:])]
    ref_o, _ = fc.reference_attention_row(sc, vv)
    acc = fc.empty_state(4)
    for p in parts:
        acc = my_merge(acc, p)
    check("my_merge folded left to right == reference", np.allclose(acc.o / acc.l, ref_o, atol=1e-12))
    tree = my_merge(my_merge(parts[3], parts[2]), my_merge(parts[1], parts[0]))
    check("my_merge in another order and tree shape == reference", np.allclose(tree.o / tree.l, ref_o, atol=1e-12))
    check("two empty states: l = 0, no NaN", probe.l == 0.0 and not np.isnan(probe.o).any())

In [ ]:
rng = np.random.default_rng(0)
keys, vals, q = rng.normal(size=(12, 16)), rng.normal(size=(12, 8)), rng.normal(size=16) * 2
scores = keys @ q / 4.0
pieces = [fc.block_state(scores[i:i + 3], vals[i:i + 3]) for i in range(0, 12, 3)]

seq = fc.empty_state(8)
for p in pieces:                                   # left to right: one CTA's inner loop
    seq = fc.merge(seq, p)
tree = fc.merge(fc.merge(pieces[0], pieces[1]), fc.merge(pieces[2], pieces[3]))   # a reduction tree
rev = fc.empty_state(8)
for p in reversed(pieces):                         # right to left: FA2 walks K/V blocks backwards
    rev = fc.merge(p, rev)
ref, _ = fc.reference_attention_row(scores, vals)
for name, st in (("sequential", seq), ("tree", tree), ("reversed", rev)):
    check(f"{name} merge == reference", np.allclose(st.finalize()[0], ref, atol=1e-12))
empty = fc.merge(fc.empty_state(8), fc.empty_state(8))
check("merging two empty states: l = 0, no NaN (the -inf guard)", empty.l == 0.0 and not np.isnan(empty.o).any())

## 3. An FA2-order tiled forward pass with causal skipping (deep dive, sections 3 and 4)

One "CTA" per Q block (the outer loop is the grid), K/V blocks inner, walked from the last block down so that the
masked diagonal tiles come first. Ragged lengths, `B_r ≠ B_c`, and rows that are fully masked inside the first tile
they visit are all exercised. The tile counts must match `fa_calculators.causal_tiles()`.

In [ ]:
def naive(Q, K, V, causal=False):
    S = (Q @ K.T) / math.sqrt(Q.shape[1])
    if causal:
        S = np.where(np.tril(np.ones(S.shape, dtype=bool)), S, -np.inf)
    mx = S.max(axis=1, keepdims=True)
    P = np.exp(S - mx)
    return (P / P.sum(axis=1, keepdims=True)) @ V, (mx[:, 0] + np.log(P.sum(axis=1)))


def fa2_forward(Q, K, V, Br, Bc, causal=False):
    N, d = Q.shape
    tau = 1.0 / math.sqrt(d)
    O, L = np.zeros_like(Q), np.zeros(N)
    visited = masked = 0
    for m in range(math.ceil(N / Br)):                               # grid axis: one CTA per Q block
        r0, r1 = m * Br, min((m + 1) * Br, N)
        mi, li, acc = np.full(r1 - r0, -np.inf), np.zeros(r1 - r0), np.zeros((r1 - r0, V.shape[1]))
        n_max = math.ceil(N / Bc)
        if causal:
            n_max = min(n_max, math.ceil((m + 1) * Br / Bc))          # stop at the diagonal
        for n in reversed(range(n_max)):                             # diagonal (masked) tiles first
            c0, c1 = n * Bc, min((n + 1) * Bc, N)
            S = (Q[r0:r1] @ K[c0:c1].T) * tau
            visited += 1
            if causal and c1 - 1 > r0:                               # tile crosses the diagonal
                masked += 1
                S = np.where(np.arange(c0, c1)[None, :] <= np.arange(r0, r1)[:, None], S, -np.inf)
            m_new = np.maximum(mi, S.max(axis=1))
            m_safe = np.where(np.isneginf(m_new), 0.0, m_new)         # guard: no (-inf) - (-inf)
            alpha = np.exp(np.where(np.isneginf(mi), -np.inf, mi - m_safe))
            P = np.exp(S - m_safe[:, None])
            li = alpha * li + P.sum(axis=1)
            acc = alpha[:, None] * acc + P @ V[c0:c1]
            mi = m_new
        O[r0:r1] = acc / li[:, None]                                 # normalize once
        L[r0:r1] = mi + np.log(li)                                   # natural-log LSE of scaled scores
    return O, L, visited, masked


rng = np.random.default_rng(1)
for N, Br, Bc in ((300, 64, 32), (512, 128, 64), (257, 32, 64)):
    Q, K, V = (rng.normal(size=(N, 64)) for _ in range(3))
    for causal in (False, True):
        O, L, visited, masked = fa2_forward(Q, K, V, Br, Bc, causal)
        O_ref, L_ref = naive(Q, K, V, causal)
        expected = fc.causal_tiles(N, N, Br, Bc)[0] if causal else math.ceil(N / Br) * math.ceil(N / Bc)
        full = math.ceil(N / Br) * math.ceil(N / Bc)
        check(f"N={N} Br={Br} Bc={Bc} causal={causal!s:5}: O, LSE match; {visited}/{full} tiles visited",
              np.allclose(O, O_ref, atol=1e-10) and np.allclose(L, L_ref, atol=1e-10) and visited == expected)

v, mk, full = fc.causal_tiles(512, 512, 128, 64)
print(f"\nthe diagram in section 4.4: N=512, 128 x 64 tiles -> {v} of {full} visited, {mk} masked")
for N in (4096, 32768):
    v, mk, full = fc.causal_tiles(N, N, 128, 64)
    print(f"N={N:>6}: {v} of {full} tiles ({v/full:.1%}), {mk} masked")

### Exercise 3: predict the causal tile count

Now the tiles are the other way round: `N = 1,024`, `B_r = 64` query rows by `B_c = 128` keys, causal. How many
tiles does the FA2-order loop visit, how many of those need the diagonal mask, and out of how many in the full
grid? Use the loop bound `n_max = ceil((m + 1)·B_r / B_c)` from the cell above. (Why does every diagonal tile get
visited by two Q blocks here?)

In [ ]:
tiles_visited = None
tiles_masked = None
tiles_full_grid = None

want = fc.causal_tiles(1024, 1024, 64, 128)
for name, got, w in zip(("visited", "masked", "full grid"), (tiles_visited, tiles_masked, tiles_full_grid), want):
    grade(f"causal tiles {name}, N = 1,024, 64 x 128", got, w, rel=0)

## 4. The backward pass from `O` and the log-sum-exp (deep dive, section 3.5)

FA2's order: one "CTA" per K/V block, Q blocks inner, `P` recomputed as `exp(S − L)`, `D = rowsum(dO ∘ O)` from a
pre-pass, and `dQ` accumulated across CTAs (on the GPU with `atomicAdd`). Checked against the dense analytic gradient
and a finite difference.

### Exercise 4: one backward tile from the saved LSE

Before reading the worked backward below: given one tile of scaled, masked scores `S` (rows `r0:r1`, columns
`c0:c1`), the saved log-sum-exp `L` of those rows, `dP = dO·Vᵀ` for the tile, and the rows of `O` and `dO`, return
the tile of `dS`. Use only what the forward saved: `P = exp(S − L)` (no max, no sum) and `D = rowsum(dO ∘ O)`.

In [ ]:
def ds_tile(S, L_rows, dP, O_rows, dO_rows):
    # YOUR CODE: two lines for P and D, one for dS = P * (dP - D)
    return None


rng = np.random.default_rng(5)
Nt, dt = 200, 32
Qt, Kt, Vt, dOt = (rng.normal(size=(Nt, dt)) for _ in range(4))
Ot, Lt, _, _ = fa2_forward(Qt, Kt, Vt, 64, 32, causal=True)
r0, r1, c0, c1 = 64, 128, 32, 64
S_full = np.where(np.tril(np.ones((Nt, Nt), bool)), (Qt @ Kt.T) / math.sqrt(dt), -np.inf)
P_full = np.exp(S_full - S_full.max(1, keepdims=True))
P_full /= P_full.sum(1, keepdims=True)
dP_full = dOt @ Vt.T
dS_dense = P_full * (dP_full - (dP_full * P_full).sum(1, keepdims=True))
got = ds_tile(S_full[r0:r1, c0:c1], Lt[r0:r1], dP_full[r0:r1, c0:c1], Ot[r0:r1], dOt[r0:r1])
grade("dS tile from (S, L, dP, O, dO) == dense softmax gradient", got, dS_dense[r0:r1, c0:c1], rel=1e-9)

In [ ]:
def dense_grads(Q, K, V, dO, causal):
    tau = 1 / math.sqrt(Q.shape[1])
    S = (Q @ K.T) * tau
    if causal:
        S = np.where(np.tril(np.ones(S.shape, dtype=bool)), S, -np.inf)
    P = np.exp(S - S.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    dP = dO @ V.T
    dS = P * (dP - (dP * P).sum(1, keepdims=True))
    return tau * dS @ K, tau * dS.T @ Q, P.T @ dO


def fa_backward(Q, K, V, O, L, dO, Br, Bc, causal):
    N, d = Q.shape
    tau = 1 / math.sqrt(d)
    D = (dO * O).sum(axis=1)                                   # the O(Nd) pre-pass
    dQ, dK, dV = np.zeros_like(Q), np.zeros_like(K), np.zeros_like(V)
    for n in range(math.ceil(N / Bc)):                         # one CTA per K/V block
        c0, c1 = n * Bc, min((n + 1) * Bc, N)
        for m in range(math.ceil(N / Br)):
            r0, r1 = m * Br, min((m + 1) * Br, N)
            if causal and c0 > r1 - 1:
                continue                                        # tile entirely above the diagonal
            S = (Q[r0:r1] @ K[c0:c1].T) * tau
            if causal:
                S = np.where(np.arange(c0, c1)[None, :] <= np.arange(r0, r1)[:, None], S, -np.inf)
            P = np.exp(S - L[r0:r1, None])                      # recomputed: no max, no sum
            dV[c0:c1] += P.T @ dO[r0:r1]
            dS = P * (dO[r0:r1] @ V[c0:c1].T - D[r0:r1, None])
            dQ[r0:r1] += tau * dS @ K[c0:c1]                    # atomicAdd on the GPU
            dK[c0:c1] += tau * dS.T @ Q[r0:r1]
    return dQ, dK, dV


rng = np.random.default_rng(2)
N, d = 200, 32
Q, K, V, dO = (rng.normal(size=(N, d)) for _ in range(4))
for causal in (False, True):
    O, L, _, _ = fa2_forward(Q, K, V, 64, 32, causal)
    got, want = fa_backward(Q, K, V, O, L, dO, 64, 32, causal), dense_grads(Q, K, V, dO, causal)
    check(f"tiled backward == dense gradients (causal={causal})",
          all(np.allclose(g_, w_, atol=1e-10) for g_, w_ in zip(got, want)))

eps, (i, j) = 1e-6, (5, 3)
loss = lambda Qx: float((naive(Qx, K, V, True)[0] * dO).sum())
Qp, Qm = Q.copy(), Q.copy()
Qp[i, j] += eps
Qm[i, j] -= eps
fd = (loss(Qp) - loss(Qm)) / (2 * eps)
O, L, _, _ = fa2_forward(Q, K, V, 64, 32, True)
check(f"finite difference dQ[{i},{j}] = {fd:.6f} matches", abs(fd - fa_backward(Q, K, V, O, L, dO, 64, 32, True)[0][i, j]) < 1e-5)

print(f"\nsaved for backward at N=32k, 32 heads: LSE {32*32768*4/1e6:.1f} MB vs P in bf16 {32*32768**2*2/1e9:.1f} GB")
print(f"backward / forward FLOPs: {fc.attention_flops(4096, 4096, 128, pass_='bwd') / fc.attention_flops(4096, 4096, 128):.1f}"
      " (5 GEMMs vs 2)")

## 5. Split-KV decode with GQA packing (deep dive, section 6)

The `g = 4` query heads that share one KV head form a 4-row tile. Each split (one CTA on the GPU) reads its slice of
K/V once for all 4 rows and writes `(Ô, LSE)`; a combine step merges them (section 2.3). Then the split counts FA2's
heuristic would choose, and how decode time grows with batch × context.

In [ ]:
rng = np.random.default_rng(7)
g, L_ctx, d = 4, 5000, 128
q = rng.normal(size=(g, d))
Kc, Vc = rng.normal(size=(L_ctx, d)), rng.normal(size=(L_ctx, d))
tau = 1 / math.sqrt(d)


def split_kv_packed(q, K, V, n_splits):
    bounds = np.linspace(0, K.shape[0], n_splits + 1).astype(int)
    outs, lses = [], []
    for lo, hi in zip(bounds[:-1], bounds[1:]):       # one CTA per split
        S = (q @ K[lo:hi].T) * tau                     # a g-row tile: K/V read once for all g heads
        m = S.max(axis=1)
        P = np.exp(S - m[:, None])
        l = P.sum(axis=1)
        outs.append((P @ V[lo:hi]) / l[:, None])
        lses.append(m + np.log(l))
    outs, lses = np.stack(outs), np.stack(lses)       # fp32 partials in HBM on the GPU
    top = lses.max(axis=0)
    lse = top + np.log(np.exp(lses - top).sum(axis=0))  # combine kernel
    return (np.exp(lses - lse)[:, :, None] * outs).sum(axis=0), lse


S_ref = (q @ Kc.T) * tau
P_ref = np.exp(S_ref - S_ref.max(1, keepdims=True))
O_ref = (P_ref / P_ref.sum(1, keepdims=True)) @ Vc
for n_splits in (1, 3, 29, 128):
    check(f"split-KV decode with {n_splits:3d} splits == reference",
          np.allclose(split_kv_packed(q, Kc, Vc, n_splits)[0], O_ref, atol=1e-12))

print(f"\nK/V bytes per KV head (bf16): packed {2*L_ctx*d*2/1e6:.2f} MB; one query head at a time {g*2*L_ctx*d*2/1e6:.2f} MB")
print(f"decode intensity: MHA {fc.decode_intensity(1):.0f}, GQA g=4 {fc.decode_intensity(4):.0f}, "
      f"MLA absorbed {fc.mla_decode_intensity():.0f} (FP8 latent {fc.mla_decode_intensity(b=1):.0f}) FLOP/B")

print(f"\n{'batch x ctx':>14} {'GPU':>5} {'CTAs':>5} {'splits':>7} {'launched':>9}")
for B, ctx in ((1, 32768), (8, 32768), (64, 4096), (4, 131072)):
    for dev in ("H100", "A100", "L4"):
        r = fc.fa2_decode_splits(B, 32, 8, ctx, 128, fc.DEVICES[dev].sms)
        print(f"{B:>4} x {ctx:>7} {dev:>5} {r['ctas_without_split']:>5} {r['splits']:>7} {r['ctas']:>9}")
check("H100, batch 1 x 32k: 8 CTAs -> 29 splits", fc.fa2_decode_splits(1, 32, 8, 32768, 128, 132)["splits"] == 29)

w_bytes, bw = 8.03e9 * 2, fc.DEVICES["H100"].hbm_tbs * 1e12       # Llama-3-8B shape, bf16, H100
print(f"\nweights alone: {w_bytes/bw*1e3:.2f} ms per step")
for B, ctx in ((1, 2048), (1, 32768), (1, 131072), (32, 2048), (32, 8192), (8, 32768)):
    kv = fc.decode_attention_bytes(B * ctx, 32, 8, 128)
    print(f"{B:>3} x {ctx:>6}: KV {kv/1e9:6.2f} GB, {kv/bw*1e3:6.2f} ms, attention share of step bytes {kv/(kv+w_bytes):6.1%}")

In [ ]:
# The same decode shapes, FA2's heuristic against FA3's as vLLM calls it on an H100 (deep dive section 6.3)
print(f"{'batch x ctx':>14} {'FA2 splits -> CTAs':>19} {'FA3 (vLLM) splits -> CTAs':>27}")
for B, ctx in ((1, 32768), (8, 32768), (64, 4096), (4, 131072)):
    r2 = fc.fa2_decode_splits(B, 32, 8, ctx, 128, 132)
    r3 = fc.fa3_decode_splits(B, 32, 8, ctx, 128, 132)
    print(f"{B:>4} x {ctx:>7} {r2['splits']:>9} -> {r2['ctas']:<7} {r3['splits']:>13} -> {r3['ctas']:<7}")
check("H100, batch 1 x 32k: FA3 in vLLM picks 15 splits (120 CTAs)", fc.fa3_decode_splits(1, 32, 8, 32768, 128, 132)["ctas"] == 120)
p = fc.split_partials_bytes(29, 1, 8, 4, 128)
print(f"\nFA2 partials at 1 x 32k: {p['o_bytes']:,} B of O + {p['lse_bytes']:,} B of LSE, "
      f"{p['traffic'] / fc.decode_attention_bytes(32768, 1, 8, 128):.2%} of one layer's K/V read")

### Exercise 5: predict FA2's split count

A Llama-3-8B-shaped decode step (32 query heads, 8 KV heads, `d = 128`) with batch 2 and 16k tokens of context per
sequence, on an L4 (58 SMs). FA2 packs the GQA group, uses 128-key blocks in its split kernel, counts `2 × SMs`
slots, and picks the smallest split count whose wave efficiency is within 85% of the best (deep dive section 6.3).
How many CTAs are there without splitting, and how many splits does it choose?

In [ ]:
ctas_without_split = None
splits = None

r = fc.fa2_decode_splits(2, 32, 8, 16384, 128, fc.DEVICES["L4"].sms)
grade("CTAs without split, 2 x 16k on an L4", ctas_without_split, r["ctas_without_split"], rel=0)
grade("FA2 splits, 2 x 16k on an L4", splits, r["splits"], rel=0)

## 6. FP8 numerics (deep dive, sections 5.6 and 9.4)

Four separate error sources:

1. **Mantissa rounding.** e4m3 keeps 3 mantissa bits, so every element carries up to 6.25% relative error. No
   scaling fixes that; only a wider format does.
2. **An outlier channel.** A per-tensor scale is set by the largest value. For integer formats (INT8/INT4 KV caches)
   the step is `amax/127`, so an outlier channel costs every other value its precision. For any format there is a
   second effect: rounding error is relative, so the error of `q·k = Σ q_i k_i` scales with `sqrt(Σ q_i² k_i²)`. If
   one channel is large in *both* Q and K, `q·k` is essentially one product and carries its full relative error.
   A random Hadamard rotation leaves `QKᵀ` unchanged and spreads the product over all `d` channels.
3. **A few outlier tokens.** A few large rows also set a per-tensor scale. One scale per block of rows (FA3's block
   quantization) confines the damage to their block.
4. **Small probabilities.** `P ≤ 1`, and e4m3's smallest subnormal is `2^-9`. FA3 multiplies `P` by `2^8` before the
   conversion (`Max_offset = 8`) and divides the row sum by the same factor.

In [ ]:
rng = np.random.default_rng(0)
N, d = 256, 128
q0, k0 = rng.normal(size=(N, d)), rng.normal(size=(N, d))
Mrot = fc.random_hadamard(d, seed=0)


def int_quant(bits):
    top = 2 ** (bits - 1) - 1
    return lambda x: np.round(x / (np.abs(x).max() / top)) * (np.abs(x).max() / top)


fp8 = lambda x: fc.quantize_fp8(x)[0]
QUANT = (("fp8 e4m3", fp8), ("int8", int_quant(8)), ("int4", int_quant(4)))


def rel_err(quant, qq, kk, ref):
    return np.linalg.norm(quant(qq) @ quant(kk).T - ref) / np.linalg.norm(ref)


res = {}
for case in ("outlier channel in K only", "same outlier channel in Q and K"):
    q, k = q0.copy(), k0.copy()
    k[:, 7] *= 20.0
    if case.startswith("same"):
        q[:, 7] *= 20.0
    ref = q @ k.T
    check(f"{case}: the rotation leaves QK^T unchanged", np.allclose((q @ Mrot) @ (k @ Mrot).T, ref))
    print(f"  amax/rms of K: plain {np.abs(k).max() / np.sqrt((k**2).mean()):.1f}, "
          f"rotated {np.abs(k @ Mrot).max() / np.sqrt(((k @ Mrot)**2).mean()):.1f}")
    for name, quant in QUANT:
        res[case, name] = (rel_err(quant, q, k, ref), rel_err(quant, q @ Mrot, k @ Mrot, ref))
        print(f"  {name:>9}, per-tensor scale: QK^T relative error {res[case, name][0]:6.2%} plain, "
              f"{res[case, name][1]:6.2%} rotated")
k_only, shared = "outlier channel in K only", "same outlier channel in Q and K"
check("K only: rotation cuts the int8 error by more than 3x", res[k_only, "int8"][1] < res[k_only, "int8"][0] / 3)
check("K only: rotation barely moves the e4m3 error (it is mantissa rounding)",
      abs(res[k_only, "fp8 e4m3"][1] - res[k_only, "fp8 e4m3"][0]) < 0.005)
check("Q and K share the channel: rotation cuts the e4m3 error by more than 5x",
      res[shared, "fp8 e4m3"][1] < res[shared, "fp8 e4m3"][0] / 5)

# outlier tokens: one scale per tensor vs one scale per 64 rows (block quantization)
k_tok = k0.copy()
k_tok[:4] *= 50.0                                           # 4 of 256 tokens
ref_tok = q0 @ k_tok.T
blockwise = lambda f: (lambda x: np.concatenate([f(x[i:i + 64]) for i in range(0, len(x), 64)]))
rest = slice(64, None)                                      # scores against the other tokens
tok = {}
for name, quant in QUANT[:2]:
    per_tensor = np.linalg.norm((quant(q0) @ quant(k_tok).T - ref_tok)[:, rest]) / np.linalg.norm(ref_tok[:, rest])
    per_block = (np.linalg.norm((blockwise(quant)(q0) @ blockwise(quant)(k_tok).T - ref_tok)[:, rest])
                 / np.linalg.norm(ref_tok[:, rest]))
    tok[name] = (per_tensor, per_block)
    print(f"outlier tokens, {name:>9}: error on the other tokens' scores {per_tensor:6.2%} per-tensor scale, "
          f"{per_block:6.2%} one scale per 64 rows")
check("outlier tokens: per-block scales cut the int8 error by more than 10x", tok["int8"][1] < tok["int8"][0] / 10)
check("outlier tokens: e4m3 barely cares (its range spans ~2^15)", abs(tok["fp8 e4m3"][1] - tok["fp8 e4m3"][0]) < 0.005)

rng = np.random.default_rng(1)
s = rng.normal(size=4096) * 2.0                             # one row of scaled scores
p = np.exp(s - s.max())
v = rng.normal(size=(4096, 64))
o_ref = p @ v / p.sum()
flushed = {}
for offset in (0, 8):
    pq = fc.round_to_e4m3(p * 2.0**offset) / 2.0**offset   # P as it enters the P.V MMA
    o = pq @ v / p.sum()                                    # the row sum l is accumulated in fp32 from unrounded p
    flushed[offset] = np.mean(pq == 0)
    print(f"P in e4m3, offset 2^{offset}: {flushed[offset]:6.1%} of probabilities flush to zero, "
          f"probability mass lost {1 - pq.sum() / p.sum():+.3%}, output error {np.linalg.norm(o - o_ref) / np.linalg.norm(o_ref):.2%}")
check("the 2^8 offset keeps small probabilities out of the flush-to-zero range", flushed[8] < flushed[0] / 10)

### Exercise 6: pick the FP8 settings

(a) FA3 scales `P` by `2^k` before rounding it to e4m3. What is the largest integer `k` that can never saturate
(e4m3's largest value is 448, and `P ≤ 1`)? What happens to the largest probability at `k + 1`?

(b) Your K cache has one outlier channel; Q does not. Your colleague proposes a Hadamard rotation of Q and K to cut
the e4m3 error of `QKᵀ`. Answer `"yes"` if it will help much, `"no"` if not.

In [ ]:
my_offset = None          # an integer
rotation_helps = None     # "yes" or "no"

want_offset = max(k for k in range(16) if 2.0**k <= fc.E4M3_MAX)
grade("largest safe P offset", my_offset, want_offset, rel=0)
if my_offset is not None:
    top = fc.round_to_e4m3(np.array([2.0 ** (my_offset + 1)]))[0] / 2.0 ** (my_offset + 1)
    print(f"      at 2^{my_offset + 1} the largest probability becomes {top:.3f}: saturated, a {1 - top:.1%} error")
if rotation_helps is None:
    print("[ ]   rotation, outlier channel in K only: not attempted yet")
else:
    want = "yes" if res[k_only, "fp8 e4m3"][1] < res[k_only, "fp8 e4m3"][0] / 2 else "no"
    assert rotation_helps == want, "look at the 'outlier channel in K only' rows above"
    print("PASS  rotation, outlier channel in K only")

## 7. Serving bookkeeping: pages, bottom-right masks, cascades (deep dive, sections 7.1 to 7.4)

Three things a serving kernel does that the prefill kernel above does not, each checked against dense attention:

1. **Paged K/V.** The cache is a pool of fixed-size blocks; a per-sequence block table maps logical block
   `t // block_size` to a physical block. The kernel walks the table one block at a time and merges states.
2. **Bottom-right causal alignment.** A chunk of `N_q` new tokens appended to a cache of `N_k − N_q` tokens: query `i`
   may see key `j` if `j ≤ i + (N_k − N_q)`.
3. **Cascade attention.** A prefix shared by every request is attended once for all of their queries, each request's
   own suffix separately, and the two partial results merge through their LSEs.

In [ ]:
rng = np.random.default_rng(3)
bs, pool_blocks, dh = 16, 64, 32
k_pool, v_pool = rng.normal(size=(pool_blocks, bs, dh)), rng.normal(size=(pool_blocks, bs, dh))
seq_lens = [37, 5, 64]
free = list(rng.permutation(pool_blocks))
block_table = [[int(free.pop()) for _ in range(math.ceil(L / bs))] for L in seq_lens]    # scattered pages


def paged_decode(q, table, L):
    """One query row against a paged cache: one block per step, merged into (m, l, o)."""
    st = fc.empty_state(dh)
    for lb, pb in enumerate(table):
        n = min(bs, L - lb * bs)                            # the last page is partly filled
        st = fc.merge(st, fc.block_state((k_pool[pb, :n] @ q) / math.sqrt(dh), v_pool[pb, :n]))
    return st.finalize()[0]


for s_idx, L in enumerate(seq_lens):
    q = rng.normal(size=dh)
    K_dense = np.concatenate([k_pool[pb] for pb in block_table[s_idx]])[:L]      # logical token t -> page, row
    V_dense = np.concatenate([v_pool[pb] for pb in block_table[s_idx]])[:L]
    ref, _ = fc.reference_attention_row(K_dense @ q, V_dense, 1 / math.sqrt(dh))
    check(f"sequence {s_idx} (L = {L}, pages {block_table[s_idx]}): paged == dense",
          np.allclose(paged_decode(q, block_table[s_idx], L), ref, atol=1e-12))

n_q, n_k = 2, 5
allowed = (np.arange(n_k)[None, :] <= np.arange(n_q)[:, None] + (n_k - n_q)).astype(int)
print("\nbottom-right causal mask, N_q = 2, N_k = 5:\n", allowed)
check("bottom-right mask is 1 1 1 1 0 / 1 1 1 1 1", (allowed == [[1, 1, 1, 1, 0], [1, 1, 1, 1, 1]]).all())
check("fc.causal_fraction(2, 5) counts the same 9 of 10 pairs", fc.causal_fraction(2, 5) == 9 / 10)

prefix_len, suffix_lens = 300, [7, 50, 1, 120]
Kp, Vp = rng.normal(size=(prefix_len, dh)), rng.normal(size=(prefix_len, dh))
qs = rng.normal(size=(len(suffix_lens), dh))                # one decode query per request
Sp = (qs @ Kp.T) / math.sqrt(dh)                            # the prefix: ONE pass for all requests
mp = Sp.max(1)
lp = np.exp(Sp - mp[:, None]).sum(1)
Op = (np.exp(Sp - mp[:, None]) @ Vp) / lp[:, None]
Lp = mp + np.log(lp)
for r, n in enumerate(suffix_lens):
    Ks, Vs = rng.normal(size=(n, dh)), rng.normal(size=(n, dh))
    o_s, l_s = fc.block_state((Ks @ qs[r]) / math.sqrt(dh), Vs).finalize()
    o, _ = fc.merge_lse(Op[r], Lp[r], o_s, l_s)
    ref, _ = fc.reference_attention_row(np.concatenate([Kp, Ks]) @ qs[r], np.concatenate([Vp, Vs]), 1 / math.sqrt(dh))
    check(f"request {r}: cascade (shared prefix + own suffix of {n}) == full attention", np.allclose(o, ref, atol=1e-12))
print(f"prefix K/V read once instead of {len(suffix_lens)} times: "
      f"{prefix_len * 2 * dh * 2:,} B instead of {len(suffix_lens) * prefix_len * 2 * dh * 2:,} B per KV head (bf16)")

## 8. The page's remaining numbers

Every derived number of the deep dive that the cells above do not already print, by section, from the calculators.

In [ ]:
MB, GB = 1e6, 1e9
h100, l4 = fc.DEVICES["H100"], fc.DEVICES["L4"]
print("1.2  per-kernel bytes, N = 4,096, d = 128:",
      {k: f"{v / MB:.2f} MB" for k, v in fc.naive_traffic(4096, 128)["passes"].items()})
print(f"     +1 elementwise pass: +{2 * 4096**2 * 2 / MB:.1f} MB; fp32 scores: "
      f"{fc.naive_traffic(4096, 128, b_s=4)['bytes'] / MB:.1f} MB")
for n in (4096, 32768):
    t = fc.naive_traffic(n, 128)
    rh, rl = fc.roofline(t["flops"], t["bytes"], "H100"), fc.roofline(t["flops"], t["bytes"], "L4")
    c = fc.compulsory_bytes(n, 128)
    print(f"1.4  N = {n:>6}: {t['flops'] / 1e9:.2f} GFLOP, S alone {t['s_matrix_bytes'] / MB:,.1f} MB; "
          f"H100 {rh['attainable_tflops']:.0f} TFLOP/s ({rh['fraction_of_peak']:.0%}), "
          f"L4 {rl['attainable_tflops']:.1f} TFLOP/s ({rl['fraction_of_peak']:.0%})")
    print(f"1.5  compulsory {c / MB:.1f} MB, intensity {t['flops'] / c:,.0f} FLOP/B")
print(f"1.4  S for a 32-head layer at 32k: {32 * 32768**2 * 2 / GB:.1f} GB")
print(f"3.2  FA1 block sizes on an H100 (116,736 bf16 elements): B_c, B_r = {fc.fa1_block_sizes(116_736, 128)}")
print(f"3.3  constant-aware FA1 saving M/(3d^2): d=128 {fc.io_saving(128, 'fa1', 116_736):.2f}x, "
      f"d=64 {fc.io_saving(64, 'fa1', 116_736):.1f}x; FA2 order 2B_r/d at d=128: {fc.io_saving(128, 'fa2', block_m=128):.0f}x; "
      f"T4 (64 KB) at d=128: {fc.io_saving(128, 'fa1', 32_768):.2f}x")
for n in (4096, 32768):
    rows = {"naive": fc.naive_traffic(n, 128),
            "FA1 B_c=128": fc.flash_traffic(n, 128, 128, 128, schedule="fa1"),
            "FA1 B_c=228": fc.flash_traffic(n, 128, 128, 228, schedule="fa1"),
            "FA2 B_r=128": fc.flash_traffic(n, 128, 128, 64),
            "FA2 causal": fc.flash_traffic(n, 128, 128, 64, causal=True)}
    print(f"3.4  N = {n:>6}: " + "; ".join(f"{k} {v['bytes'] / MB:,.1f} MB / {v['intensity']:.0f}" for k, v in rows.items()))
print(f"3.3  d = 64, N = 4,096: naive {fc.naive_traffic(4096, 64)['bytes'] / MB:.1f} MB, "
      f"FA1 B_c=456 {fc.flash_traffic(4096, 64, 64, 456, schedule='fa1')['bytes'] / MB:.1f} MB")
g = fc.attention_flops(4096, 4096, 128) / 2
print(f"3.6  recompute GEMM {g / 1e9:.1f} GFLOP = {g / (h100.bf16_tflops * 1e12) * 1e6:.1f} us on an H100; "
      f"writing + reading P {2 * 4096**2 * 2 / MB:.0f} MB = {2 * 4096**2 * 2 / (h100.hbm_tbs * 1e12) * 1e6:.0f} us")
for dev in ("A100", "H100", "B200"):
    c = fc.clocks_per_score(dev)
    print(f"4.3  {dev}: MMA {c['mma']:.4f}, EX2 {c['ex2']:.4f} ({c['ex2_vs_mma']:.0%}), "
          f"FP32 {c['fp32']:.3f} ({c['fp32_vs_mma']:.1%}) clocks per score")
print(f"5.2  FA3 registers: {fc.fa3_registers()}")
L = 32768
print(f"6.1  one Llama-3-8B layer at L = 32k: K/V {fc.decode_attention_bytes(L, 1, 8, 128) / MB:.0f} MB, "
      f"{4 * L * 128 * 4 * 8 / 1e6:.0f} MFLOP, {fc.decode_attention_bytes(L, 1, 8, 128) / (h100.hbm_tbs * 1e12) * 1e6:.0f} us; "
      f"32 layers {fc.decode_attention_bytes(L, 32, 8, 128) / (h100.hbm_tbs * 1e12) * 1e3:.2f} ms")
print(f"6.5  KV bytes = weight bytes at {8.03e9 * 2 / fc.kv_bytes_per_token(32, 8, 128):,.0f} cached tokens; "
      f"L4 times are {h100.hbm_tbs / l4.hbm_tbs:.1f}x the H100's")
print(f"8    sliding window W = 4,096 at 32k, 128 x 64 tiles: {fc.window_tiles(32768, 4096, 128, 64):,} of "
      f"{fc.causal_tiles(32768, 32768, 128, 64)[0]:,} causal tiles "
      f"({fc.window_tiles(32768, 4096, 128, 64) / fc.causal_tiles(32768, 32768, 128, 64)[0]:.0%})")
r = fc.mla_cache_ratio()
print(f"8.1  MLA: {r['mha_bytes']:,} vs {r['mla_bytes']:,} B per token per layer = {r['ratio']:.1f}x "
      f"({fc.mla_cache_ratio(count_rope_in_key=False)['ratio']:.1f}x without the rotary dims); decode intensity "
      f"{fc.mla_decode_intensity():.0f} (ridge {h100.ridge:.0f}, {fc.mla_decode_intensity() / h100.ridge:.0%}), "
      f"FP8 latent {fc.mla_decode_intensity(b=1):.0f}, TP=8 {fc.mla_decode_intensity(n_heads=16):.0f}; "
      f"absorbed prefill {fc.mla_prefill_absorbed_ratio():.1f}x the quadratic FLOPs")
w = fc.padding_waste([100, 3000, 500])
print(f"8.2  padding [100, 3000, 500]: {w['padded_pairs'] / 1e6:.1f} M vs {w['actual_pairs'] / 1e6:.2f} M pairs "
      f"({w['waste']:.1f}x); a 100-token sequence uses {w['tile_utilisation'][0]:.0%} of its tile")
print(f"8.4  ring attention, tokens per GPU to hide the hop: NVLink {fc.ring_min_tokens_per_gpu(989.4, 450):,.0f} "
      f"(peak) / {fc.ring_min_tokens_per_gpu(600, 450):,.0f} (600 TFLOP/s); 50 GB/s "
      f"{fc.ring_min_tokens_per_gpu(600, 50):,.0f} to {fc.ring_min_tokens_per_gpu(989.4, 50):,.0f}")
print(f"10.5 A100 at 50-73%: {0.50 * 312:.0f}-{0.73 * 312:.0f} TFLOP/s; H100 FA2 at 35%: {0.35 * 989.4:.0f}; "
      f"L4 at 50-70% (assumption): {0.5 * 121:.1f}-{0.7 * 121:.1f}")
lin = 4096 * 4096 * 2 + 4096 * 1024 * 2 + 3 * 4096 * 14336
print("10.6 attention / linear FLOPs: " + ", ".join(
    f"{n // 1024}k {fc.prefill_attention_share(n, 32, 128, lin):.2f}" for n in (2048, 8192, 32768, 131072)))
print(f"11.3 Triton defaults stage {fc.tile_smem_bytes(128, 64, 128, kv_stages=2):,} B of tiles; "
      f"T4 suggestion (64 x 32, 1 stage) {fc.tile_smem_bytes(64, 32, 128):,} B")

## In a design review

**The two-minute version.** "Naive attention writes and rereads an `N × N` matrix, so its intensity tends to `d/b`,
64 FLOP per byte at `d = 128` in bf16, far below every GPU's ridge; bigger batches do not help. FlashAttention keeps
the math and changes the schedule: a partial softmax result `(m, l, o)` merges exactly with any other, so each thread
block streams K/V tiles past a resident Q tile, keeps the output in registers, writes it once, and saves only one
log-sum-exp per row for the backward, which recomputes `P`. With the constants kept, the FA1 schedule saves about
`M/(3d²)` in traffic and the FA2 schedule `2B_r/d` before the L2 helps further. Decode is a different kernel: one
query row per head, no reuse, so it is split across thread blocks, packs the GQA group, and merges the pieces with the
same operator; serving adds block tables, bottom-right masks and cascades, all the same algebra. FP8 has four error
sources with four different fixes, and the rotation only helps e4m3 when the outliers of Q and K line up."

**Q1. Someone proposes making naive attention faster by increasing the batch size. Why won't that help?**
Batch, heads and sequence length scale FLOPs and bytes together; the naive schedule's intensity tends to `d/b`
(64 FLOP/B at `d = 128`, bf16), below every GPU's ridge. Only a schedule change (fusion and tiling) moves it.

**Q2. Split-KV decode writes partial results to HBM and adds a kernel. When is that worth it?**
When `batch × KV heads` CTAs cannot fill the SMs (a few sequences, long contexts). The partials are tiny next to the
K/V bytes (0.7% at batch 1 × 32k on an H100 with FA2's 29 splits; FA3's 15 halve that), and the extra CTAs are what
saturate bandwidth. At large batch the heuristics pick one split.

**Q3. Would a Hadamard rotation fix our FP8 attention accuracy problem?**
It depends on where the error comes from. For integer formats (INT8/INT4 KV caches) an outlier channel ruins the
scale and the rotation fixes it. For e4m3 it helps when the same channel is large in both Q and K (the dot product
is then one dominant product; section 6 shows about 8× less error) and does nothing when only one side has the
outlier, because the remaining error is the 3-bit mantissa. Outlier *tokens* need block scales, not a rotation.
Check the model's own Q and K activations, then measure end-to-end quality before and after.